1. Import necessary libraries

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import warnings
warnings.filterwarnings('ignore') # Keeps your notebook clean from annoying red text



2. Load the data + format dates

In [8]:
# Load Data
df = pd.read_csv('Full_Dataset.csv')

# Parse European dates and set as the timeline index
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y %H:%M')
df.set_index('Date', inplace=True)

# --- THE FIX: Handle Missing Values ---
# 1. Let's see how many missing values you actually had:
print("Missing values before cleaning:")
print(df.isna().sum())

# 2. Fill the missing values using linear interpolation
df.interpolate(method='linear', inplace=True)

# 3. Just in case the very first row was missing (interpolation can't look backwards)
df.fillna(method='bfill', inplace=True) 

print("\nMissing values after cleaning:")
print(df.isna().sum())
# ------------------------------------

df = df.loc['2023-01-01':] # Keep only data from 2023 onwards to leave out the high prices due to the massive historic energy crisis in europe in 2021/2022

# Let's take a quick look to make sure it worked
print(f"Total hours of data: {len(df)}")
df.head(3)

Missing values before cleaning:
Price_BE      0
Load_FR       3
Gen_FR      292
Price_CH      0
Wind_BE       5
Solar_BE      5
Load_BE       0
dtype: int64


TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

3. Data scaling and definition of the sliding window for model training 
- the target data is isolated (BE prices) 
- data is scaled between 0 and 1 (normalised)
- The sliding window is defined (= the amount of data that is used for prediction)


In [ ]:
# 1. Select the features you want to use (based on your correlation matrix!)
# Let's use Price, Belgian Load, Wind, Solar, and French Load as an example
features = ['Price_BE', 'Load_BE', 'Wind_BE', 'Solar_BE', 'Load_FR']

# Extract all these columns as a 2D array
multivariate_data = df[features].values

# 2. Scale the data (Crucial for Neural Networks)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(multivariate_data)

# 3. Define the sliding window function (Slightly updated)
def create_multivariate_sequences(data, look_back, predict_ahead):
    X, y = [], []
    for i in range(len(data) - look_back - predict_ahead + 1):
        # X now contains ALL features for the past 168 hours
        X.append(data[i : (i + look_back), :])
        # y ONLY needs the target variable (Price_BE is index 0)
        y.append(data[i + look_back : (i + look_back + predict_ahead), 0])
    return np.array(X), np.array(y)

LOOK_BACK = 168     
PREDICT_AHEAD = 72  

X, y = create_multivariate_sequences(scaled_data, LOOK_BACK, PREDICT_AHEAD)

print(f"X (Inputs) shape: {X.shape}") 
# Notice X shape is now (..., 168, 5) because we have 5 features!
print(f"y (Targets) shape: {y.shape}")

4. Data split between the data used for training vs. validation

In [ ]:
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

X_train, X_val = X[:split_index], X[split_index:]
y_train, y_val = y[:split_index], y[split_index:]

print(f"Training on {len(X_train)} windows.")
print(f"Validating on {len(X_val)} windows.")

5. Building + training of the LSTM model 

In [ ]:
model = Sequential()

# First LSTM Layer
model.add(LSTM(units=64, return_sequences=True, input_shape=(LOOK_BACK, len(features))))
model.add(Dropout(0.2)) # Drops 20% of neurons randomly to prevent overfitting

# Second LSTM Layer
model.add(LSTM(units=32, return_sequences=False))
model.add(Dropout(0.2))

# Output Layer: Must exactly match your PREDICT_AHEAD number
model.add(Dense(units=PREDICT_AHEAD))

# Compile the model using Mean Squared Error (MSE) just like the assignment asks
model.compile(optimizer='adam', loss='mse')

# Train the model! (This might take a few minutes to run)
print("Starting training...")
history = model.fit(
    X_train, y_train,
    epochs=15,          # How many times it loops through the whole dataset
    batch_size=64,      # How many flashcards it looks at before updating its math
    validation_data=(X_val, y_val),
    verbose=1
)
print("Training Complete!")

6. Check for overfitting
--> creates graph showing Training MSE vs Validation MSE (If the orange line starts going up while the blue line goes down, model is overfitting)

In [ ]:
# Cell 6: Plotting the Loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training MSE (Loss)')
plt.plot(history.history['val_loss'], label='Validation MSE (Loss)')
plt.title('Model Training History (MSE)')
plt.xlabel('Epochs (Training Rounds)')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

7. Final Test & Export 
This cell generates the actual assignment submission. It grabs the very last 168 hours of the dataset, asks the trained model to predict the next 72 hours, undoes the scaling (so the prices are back in Euros), and saves it to a CSV flawlessly.

In [ ]:
# --- UPDATED CELL 7: Generate predictions.csv (Multivariate Version) ---

# 1. Grab the very last 168 hours of ALL features (from 'scaled_data' instead of 'scaled_prices')
last_168_hours = scaled_data[-LOOK_BACK:]

# 2. Reshape it so the neural network accepts it: (1 window, 168 hours, 5 features)
last_window = last_168_hours.reshape(1, LOOK_BACK, len(features))

# 3. Make the 72-hour prediction
scaled_prediction = model.predict(last_window)

# 4. The Dummy Array Trick for Inverse Transforming
# Our scaler expects 5 columns to reverse the math. We create a blank table of zeros...
dummy_array = np.zeros((PREDICT_AHEAD, len(features)))

# ...and insert our 72 predictions into the 0th column (which is Price_BE)
dummy_array[:, 0] = scaled_prediction.flatten()

# Inverse transform the whole table, but only extract the first column (our real Euro prices!)
final_prices = scaler.inverse_transform(dummy_array)[:, 0]

# 5. Format it EXACTLY as the assignment demands: 72 rows, 1 column
submission_df = pd.DataFrame(final_prices)

# 6. Export with NO index and NO header
submission_df.to_csv('predictions.csv', index=False, header=False)

print("\nSUCCESS! Saved to predictions.csv")
print("Check the file: It should have exactly 72 rows, 1 column, and no text headers.")

8. Prediction and uncertainty

In [ ]:
# --- UPDATED CELL 8: Prediction vs Actual with Uncertainty (Multivariate Fix) ---

# 0. Helper function to use the "Dummy Array Trick" on any 1D prediction
def inverse_transform_target(scaled_1d_array):
    # Create a blank table of zeros (72 rows, 5 columns)
    dummy = np.zeros((len(scaled_1d_array), len(features)))
    # Put our scaled prices into the 0th column
    dummy[:, 0] = scaled_1d_array.flatten()
    # Inverse transform and return just the 0th column (Euros!)
    return scaler.inverse_transform(dummy)[:, 0]

# 1. Pick a random window from our validation set to test
sample_index = 100 
X_sample = X_val[sample_index : sample_index + 1]
y_actual_scaled = y_val[sample_index]

# 2. Get the Actual prices using our new helper function
y_actual = inverse_transform_target(y_actual_scaled)

# 3. Get the Standard Point Prediction
y_pred_scaled = model.predict(X_sample)
y_pred = inverse_transform_target(y_pred_scaled[0])

# --- MONTE CARLO DROPOUT FOR P10/P90 ---
print("Running Monte Carlo simulations for P10/P90...")
n_iterations = 100
mc_predictions = []

for _ in range(n_iterations):
    # Force Dropout ON
    mc_pred_scaled = model(X_sample, training=True).numpy()[0]
    # Inverse transform the simulation using the helper function
    mc_pred = inverse_transform_target(mc_pred_scaled)
    mc_predictions.append(mc_pred)

mc_predictions = np.array(mc_predictions)

# Calculate Percentiles
p10 = np.percentile(mc_predictions, 10, axis=0).flatten()
p50 = np.percentile(mc_predictions, 50, axis=0).flatten()
p90 = np.percentile(mc_predictions, 90, axis=0).flatten()

# --- PLOTTING ---
plt.figure(figsize=(14, 6))
hours = range(PREDICT_AHEAD)

plt.fill_between(hours, p10, p90, color='orange', alpha=0.3, label='P10 - P90 Uncertainty Band')
plt.plot(hours, y_actual, label='Actual Price (Ground Truth)', color='black', linewidth=2)
plt.plot(hours, y_pred, label='Standard Prediction', color='dodgerblue', linestyle='--', linewidth=2)
plt.plot(hours, p50, label='MC Dropout Median (P50)', color='darkorange', linestyle='-.')

plt.title('72-Hour Forecast vs Actuals (Multivariate Model)', fontsize=15)
plt.xlabel('Hours Ahead', fontsize=12)
plt.ylabel('Price (€/MWh)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()